# **SUBSETTING EPI CELLS WITHOUT CLUSTER C18 FOLLOWED BY PROCESSING THEM**

## Clean Environment

In [ ]:
list = ls(all.names = TRUE) # will clear all objects including hidden objects
list
rm(list)
gc()

## Load Packages

In [ ]:
packages <- c("Seurat","Signac", "here",
  "cowplot","tidyverse", "patchwork", "ggplot2", 
  "DoubletFinder",
  "clustree", "sceasy",
  "Matrix", "matrixStats",
  "Herper", "reticulate", "ggVennDiagram")

purrr::walk(packages, library, character.only = TRUE) #character.only to treat every element of 'packages' as character strings representing package names. 

# Check the version of Seurat being used here 
as.character(packageVersion("Seurat")) 

## Params

In [ ]:
params <- list(
    samples_names = c("Pilot1", "Batch1"),
    samples_directories = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/individual_preprocessing/results/sobjs",
    sample_id = "Epi_without_C18",
    res_dir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/subsetting_epi/results",
    table_dir = "tables",
    so_dir = "sobjs",
    fig_dir = "figures"
)

## Set Directories

In [ ]:
# set path to results folder, seurat objects folder and tables folder 
res_dir <- here(paste0(params$res_dir))
so_dir <- here(res_dir, params$so_dir)
table_dir <- here(res_dir, params$table_dir)
fig_dir <- here(res_dir, params$fig_dir)

In [ ]:
res_dir

## Load Object

In [ ]:
w <- readRDS("/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/integration/results/sobjs/Integration_P1-B1_seurat-wnn-integrated_annotated.RDS")

In [ ]:
DefaultAssay(w) <- "SCT"
Idents(w) <- "Annotated"
unique(Idents(w))


In [ ]:
# subset 
w_epi <- subset(w, 
                idents = c("Parietal", "Parietal_25", "Foveolar", "Neck", "Chief", "Dividing", 
                           "Tuft", "D_cells", "Chga_Hdc", "Ghrl", "Isthmus_5", "Isthmus_12", "Isthmus_20"))

In [ ]:
w_epi

In [ ]:
unique(Idents(w_epi))

In [ ]:
here(so_dir, paste0(params$sample_id))

In [ ]:
# save
saveRDS(w_epi, here(so_dir, paste0(params$sample_id, "_seurat-wnn-integrated_annotated.RDS")))

# RNA Epi

In [ ]:
# split
w_epi_list <- SplitObject(w_epi, split.by = "orig.ident2")
w_epi_list

# find most variable features across samples to integrate
int_epi <- SelectIntegrationFeatures(object.list = w_epi_list,
                                      nfeatures = 3000)
head(int_epi)


# manually set variable features of merged seurat obj
previous_var_feat <- VariableFeatures(w_epi)
setequal(previous_var_feat, int_epi)
intersect(int_epi, previous_var_feat)
identical(previous_var_feat, int_epi)

VariableFeatures(w_epi) <- int_epi

In [ ]:
# Calculate PCs using manually set variable features
w_epi <- RunPCA(w_epi, assay = "SCT", npcs = 50, reduction.name = "pca_epi", reduction.key = "PCepi_") 

In [ ]:
w_epi <- IntegrateLayers(object = w_epi,
                          method = HarmonyIntegration,
                          orig.reduction = "pca_epi",
                          normalization.method = "SCT",
                          new.reduction = "harmonySCTepi",
                          group.by = "orig.ident2")

In [ ]:
unique(w_epi@meta.data$orig.ident2)

In [ ]:
# check harmony reductions
w_epi@reductions

In [ ]:
w_epi <- RunUMAP(w_epi, dims = 1:50, slot = "data", assay = "SCT", n.neighbors = 30L, 
                 reduction = "harmonySCTepi", reduction.name = 'umap_harmonySCTepi')

In [ ]:
w_epi <- w_epi %>%
    FindNeighbors(reduction = "harmonySCTepi", dims = 1:50, graph.name = c("harmonySCTepi_nn", "harmonySCTepi_snn"), k.param = 30) %>%
    FindClusters(resolution = c(0.2, 0.5, 0.8, 1), graph.name = "harmonySCTepi_snn")

In [ ]:
# harmony visualization
e1 <- DimPlot(w_epi, reduction = "umap_harmonySCTepi", group.by = "orig.ident2", shuffle = TRUE) + labs(title = "Harmony Epithelial Cells w/o C18")
e2 <- DimPlot(w_epi, reduction = "umap_harmonySCTepi", group.by = "HTO_origin", shuffle = TRUE) + labs(title = "Harmony Epithelial Cells w/o C18")
e1 + e2
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNA_integrated-UMAPs.png")), e1 + e2, width = 20, height = 10, units = "in")

In [ ]:
# visualize clusters
cD11 <- DimPlot(w_epi, reduction = "umap_harmonySCTepi", group.by = "harmonySCTepi_snn_res.0.2", label = TRUE, pt.size = 1) + labs(title = "Harmony Epithelial Cells w/o C18 res.0.2", paste0(params$rna_merged_dim, "PCs"))
cD12 <- DimPlot(w_epi, reduction = "umap_harmonySCTepi", group.by = "harmonySCTepi_snn_res.0.5", label = TRUE, pt.size = 1) + labs(title = "Harmony Epithelial Cells w/o C18 res.0.5", paste0(params$rna_merged_dim, "PCs"))
cD13 <- DimPlot(w_epi, reduction = "umap_harmonySCTepi", group.by = "harmonySCTepi_snn_res.0.8", label = TRUE, pt.size = 1) + labs(title = "Harmony Epithelial Cells w/o C18 res.0.8", paste0(params$rna_merged_dim, "PCs"))
cD14 <- DimPlot(w_epi, reduction = "umap_harmonySCTepi", group.by = "harmonySCTepi_snn_res.1", label = TRUE, pt.size = 1) + labs(title = "Harmony Epithelial Cells w/o C18 res.1", paste0(params$rna_merged_dim, "PCs"))
cD11+cD12+cD13+cD14

ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNA_integrated-clusters-resolutions-comparisons.png")), cD11+cD12+cD13+cD14, width = 15, height = 10, units = "in")

In [ ]:
c15 <- DimPlot(w_epi, reduction = "umap_harmonySCTepi", group.by = "Annotated", label = TRUE, pt.size = 1) + labs(title = "Harmony Epithelial Cells w/o C18 - Barcodes previously annotated using whole datasets")
c15
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNA_integrated-TransferredCellAnnotationBarcodes.png")), c15, width = 15, height = 10, units = "in")

In [ ]:
w_epi

In [ ]:
# merged 
m1 <- DimPlot(w_epi, reduction = "umap", group.by = "orig.ident2", shuffle = TRUE) + labs(title = "Merged Epithelial Cells w/o C18")
m1

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNA_merged-vs-integrated-UMAP.png")), m1+e1, width = 25, height = 10, units = "in")

In [ ]:
# save rds
saveRDS(w_epi, here(so_dir, paste0(params$sample_id, "_harmony_sct.rds")))

# **ATAC Epi**

## Using previously saved merged peaks object

In [ ]:
merged_peaks_reduced <- readRDS("/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/integration/results/sobjs/Integration_P1-B1_reducedPeaks_merged_processed.rds")

In [ ]:
merged_peaks_reduced

In [ ]:
# keep only cells classified as epi cells previously
DefaultAssay(w_epi) <- "SCT"
Idents(w_epi) <- "Annotated"
epi_barcodes <- WhichCells(w_epi, idents = unique(Idents(w_epi)))
head(epi_barcodes)

In [ ]:
length(epi_barcodes)

In [ ]:
epi_merged_peaks_reduced <- subset(merged_peaks_reduced, cells = epi_barcodes)
epi_merged_peaks_reduced

In [ ]:
# Find Integration Anchors using the genomic regions created by reduce()
epi_merged_list <- SplitObject(epi_merged_peaks_reduced, split.by = "orig.ident2") 
epi_merged_list

In [ ]:
# find integration anchors with default settings
peaks.integration.anchors <- FindIntegrationAnchors(
  object.list = epi_merged_list,
  anchor.features = rownames(epi_merged_peaks_reduced),
  reduction = "rlsi",
  dims = 1:50,
  k.anchor = 5, k.filter = 200, k.score = 30,
  max.features = 200, scale = TRUE, n.trees = 50
)

In [ ]:
peaks.integration.anchors

In [ ]:
anchor.features <- peaks.integration.anchors@anchor.features
head(anchor.features)
length(anchor.features)

In [ ]:
# integrate LSI embeddings
epi_peaks_reduced_integrated <- IntegrateEmbeddings(
  anchorset = peaks.integration.anchors,
  reductions = epi_merged_peaks_reduced[['lsi']],
  new.reduction.name = "epi_integrated_lsi",
  dims.to.integrate = 2:50,
  k.weight = 100)
epi_peaks_reduced_integrated

In [ ]:
# process integrated obj
epi_peaks_reduced_integrated <- RunUMAP(epi_peaks_reduced_integrated, reduction = "epi_integrated_lsi", dims = 1:49, reduction.name = "umap_peaks_reduced_integrated_epi") %>%
  FindNeighbors(reduction = "epi_integrated_lsi", 1:49, graph.name = c("epi_peaks_reduced_integrated_nn", "epi_peaks_reduced_integrated_snn")) %>%
  FindClusters(resolution = c(0.2, 0.5, 0.8, 1), graph.name = "epi_peaks_reduced_integrated_snn")

In [ ]:
# plot integrated
epi_reduced_integrated_umap <- DimPlot(epi_peaks_reduced_integrated,
                                         group.by = "orig.ident2", shuffle = TRUE) + labs(title = "Peaks Reduced Integrated - Epi Clusters w/o C18")

epi_reduced_integrated_umap


In [ ]:
# plot integrated
epi_reduced_integrated_umap2 <- DimPlot(merged_peaks_reduced,
                                         group.by = "orig.ident2", shuffle = TRUE) + labs(title = "Peaks Reduced Merged - Epi Clusters w/o C18")

epi_reduced_integrated_umap2

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_reducedPeaks_merged-vs-integrated-UMAP.png")), epi_reduced_integrated_umap + epi_reduced_integrated_umap2, width = 25, height = 10, units = "in")


In [ ]:
# plot diff resolutions umaps
epi_peaks_reduced_integrated_umap <- DimPlot(epi_peaks_reduced_integrated, group.by = "orig.ident2", shuffle = TRUE) + labs(title = "Peaks Reduced Integrated Epi Clusters w/o C18")
epi_peaks_reduced_integrated_umap2 <- DimPlot(epi_peaks_reduced_integrated, reduction = "umap_peaks_reduced_integrated_epi", group.by = "HTO_origin", shuffle = TRUE) + labs(title = "Peaks Reduced Integrated Epi Clusters w/o C18")
epi_peaks_reduced_integrated_umap + epi_peaks_reduced_integrated_umap2 

ggsave(file.path(fig_dir, paste0(params$sample_id, "_reducedPeaks_integrated-UMAP.png")), epi_peaks_reduced_integrated_umap + epi_peaks_reduced_integrated_umap2 , width = 15, height = 7, units = "in")

In [ ]:
# visualize clusters
c21 <- DimPlot(epi_peaks_reduced_integrated, group.by = "epi_peaks_reduced_integrated_snn_res.0.2", label = TRUE, pt.size = 1) + labs(title = "reduced peaks integrated res.0.2 Epi Clusters w/o C18", paste0(params$peaks_integration_dim, "PCs"))
c22 <- DimPlot(epi_peaks_reduced_integrated, group.by = "epi_peaks_reduced_integrated_snn_res.0.5", label = TRUE, pt.size = 1) + labs(title = "reduced peaks integrated res.0.5  Epi Clusters w/o C18", paste0(params$peaks_integration_dim, "PCs"))
c23 <- DimPlot(epi_peaks_reduced_integrated, group.by = "epi_peaks_reduced_integrated_snn_res.0.8", label = TRUE, pt.size = 1) + labs(title = "reduced peaks integrated res.0.8 Epi Clusters w/o C18", paste0(params$peaks_integration_dim, "PCs"))
c24 <- DimPlot(epi_peaks_reduced_integrated, group.by = "epi_peaks_reduced_integrated_snn_res.1", label = TRUE, pt.size = 1) + labs(title = "reduced peaks integrated res.1 Epi Clusters w/o C18", paste0(params$peaks_integration_dim, "PCs"))

# DimPlot(peaks_reduced_integrated, group.by = "peaks_reduced_integrated_snn_res.1", label = TRUE, pt.size = 1) + labs(title = "Clusters reduced peaks integrated - res.1 - Healthy P1+B1", paste0(params$peaks_integration_dim, "PCs"))

c21+c22+c23+c24

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_reducedPeaks-integrated-UMAPS.png")), c21+c22+c23+c24, width = 25, height = 17, units = "in")


In [ ]:
 # checking that all barcodes are found in the previous seurat obj
length(w_epi$Annotated[colnames(w_epi) %in% colnames(epi_peaks_reduced_integrated)])

In [ ]:
# GET LABELS FROM W_EPI OBJECT
epi_peaks_reduced_integrated$Annotated <- as.character(w_epi$Annotated)
Idents(epi_peaks_reduced_integrated)
head(unique(epi_peaks_reduced_integrated$Annotated))
Idents(epi_peaks_reduced_integrated) <- "Annotated"

In [ ]:
# plot clusters
c25 <- DimPlot(epi_peaks_reduced_integrated, group.by = "Annotated", label = TRUE, pt.size = 1) + labs(title = "reduced integrated peaks Epi Cells w/o C18 \n Barcodes previously annotated using whole datasets", paste0("50 PCs"))
c25
ggsave(file.path(fig_dir, paste0(params$sample_id, "_ATAC_integrated-TransferedCellAnnotationBarcodes.png")), c25, width = 15, height = 10, units = "in")



In [ ]:
# save including barcodes annotated 
saveRDS(epi_peaks_reduced_integrated, here(so_dir, paste0(params$sample_id, "_peaksreducedintegrated.rds")))


# Epi WNN

### load in EPI RNA integrated object

In [ ]:
w_epi

In [ ]:
Graphs(w_epi)
Reductions(w_epi)

In [ ]:
w <- w_epi
w

### load in EPI ATAC integrated object

In [ ]:
ATAC <- epi_peaks_reduced_integrated

In [ ]:
ATAC

In [ ]:
Graphs(ATAC)
Reductions(ATAC)

In [ ]:
w[["reducedPeaks"]] <- ATAC[["reducedPeaks"]]
w[["epi_integrated_lsi"]] <- ATAC[["epi_integrated_lsi"]]
w[["umap_peaks_reduced_integrated_epi"]] <- ATAC[["umap_peaks_reduced_integrated_epi"]]
w[["epi_peaks_reduced_integrated_snn"]] <- ATAC[["epi_peaks_reduced_integrated_snn"]]
w

### run WNN integration

In [ ]:
# Run WNN 
w <- FindMultiModalNeighbors(
  object = w,
  reduction.list = list("pca_epi", "epi_integrated_lsi"),
  dims.list = list(1:50, 1:49),
  verbose = TRUE)

In [ ]:
Reductions(w)
Graphs(w)

In [ ]:
DefaultAssay(w) <- "SCT"

In [ ]:
w <- RunUMAP(
  object = w, 
  reduction = "pca_epi",
  slot = "data", 
  nn.name = "weighted.nn", 
  verbose = TRUE,
  reduction.name = "jointumap_epi",
  reduction.key = "epi_jointUMAP_")

In [ ]:
w <- FindClusters(
  w, 
  graph.name = "wsnn", 
  algorithm = 3, verbose = FALSE, resolution = c(0.2, 0.5, 0.8, 1))


In [ ]:
w

In [ ]:
# save objetct
saveRDS(w, here(so_dir, paste0(params$sample_id, "_seurat-wnn-integrated.RDS")))


In [ ]:
# plot RNA, ATAC and WNN
DefaultAssay(w) <- "SCT"
w1 <- DimPlot(w, reduction = "umap_harmonySCTepi", label = TRUE, label.size = 3, repel = TRUE, group.by = "Annotated") + ggtitle("Integrated RNA - Epi Clusters w/o C18")
w1

DefaultAssay(w) <- "reducedPeaks"
w2 <- DimPlot(w, reduction = "umap_peaks_reduced_integrated_epi",  label = TRUE, label.size = 3, repel = TRUE, group.by = "Annotated") + ggtitle("Integrated ATAC - Epi Clusters w/o C18")
w2

# w@meta.data$wsnn_res.1 <- factor(w@meta.data$wsnn_res.1, levels = as.character(0:max(as.numeric(w@meta.data$wsnn_res.1))))
w3 <- DimPlot(w, reduction = "jointumap_epi", label = TRUE, label.size = 3, repel = TRUE, group.by = "Annotated") + ggtitle("Integrated WNN - Epi Clusters w/o C18")
w3

w1 + w2 + w3

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_WNN_RNA_ATAC-clustering-comparison.png")), w1 + w2 + w3, width = 30, height = 7, units = "in")

In [ ]:
# plot RNA, ATAC and WNN based on orig.ident
wo1 <- DimPlot(w, reduction = "umap_harmonySCTepi", label = FALSE, label.size = 3, repel = TRUE, group.by = "HTO_origin", shuffle = TRUE) + ggtitle("Integrated RNA - Epi Clusters w/o C18")
wo2 <- DimPlot(w, reduction = "umap_peaks_reduced_integrated_epi",  label = FALSE, label.size = 3, repel = TRUE, group.by = "HTO_origin", shuffle = TRUE) + ggtitle("Integrated ATAC - Epi Clusters w/o C18")
wo3 <- DimPlot(w, reduction = "jointumap_epi", label = FALSE, label.size = 3, repel = TRUE, group.by = "HTO_origin", shuffle = TRUE) + ggtitle("Integrated WNN - Epi Clusters w/o C18")
wo1+wo2+wo3

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_WNN_RNA_ATAC-origident-comparison.png")), wo1+wo2+wo3, width = 30, height = 6, units = "in")

In [ ]:
# save all together
ggsave(file.path(fig_dir, paste0(params$sample_id, "_WNN_RNA_ATAC-umap-comparisons.png")), w1+w2+w3 + wo1+wo2+wo3, width = 30, height = 12, units = "in")

In [ ]:
# Plot diff resolutions for WNN
wc1 <- DimPlot(w, reduction = 'jointumap_epi', group.by = "wsnn_res.0.2", label = TRUE, pt.size = 0.5) + labs(title = "WNN res.0.2 - Epi Clusters w/o C18")
wc2 <- DimPlot(w, reduction = 'jointumap_epi', group.by = "wsnn_res.0.5", label = TRUE, pt.size = 0.5) + labs(title = "WNN res.0.5 - Epi Clusters w/o C18")
wc3 <- DimPlot(w, reduction = 'jointumap_epi', group.by = "wsnn_res.0.8", label = TRUE, pt.size = 0.5) + labs(title = "WNN res.0.8 - Epi Clusters w/o C18")
wc4 <- DimPlot(w, reduction = 'jointumap_epi', group.by = "wsnn_res.1", label = TRUE, pt.size = 0.5) + labs(title = "WNN res.1 - Epi Clusters w/o C18")
wc1 + wc2 + wc3 + wc4

ggsave(file.path(fig_dir, paste0(params$sample_id, "_Epi_WNN_Cluster-Resolutions.png")), wc1 + wc2 + wc3 + wc4, width = 20, height = 15, units = "in")

In [ ]:
w

In [ ]:
Graphs(w)

In [ ]:
w1 <- DietSeurat(w, 
                assays = c("SCT", "reducedPeaks"),
                graphs = c("wknn", "wsnn", "harmnoySCTepi_nn", "harmonySCTepi_snn", "epi_peaks_reduced_integrated_snn", 'harmonySCTepi_nn'),
                dimreducs = c("pca_epi", "harmonySCTepi", "jointumap_epi", "umap_harmonySCTepi","epi_integrated_lsi", "umap_peaks_reduced_integrated_epi"),
                counts = TRUE,
                data = TRUE,
                scale.data = FALSE)

In [ ]:
saveRDS(w1, here(so_dir, paste0(params$sample_id, "_Diet_seurat-wnn-integrated_annotated.RDS")))

In [ ]:
# save metadata
metadata1 <- w1@meta.data 
metadata1

In [ ]:
# Create a data frame with barcodes and their associated cell types
cell_info1 <- data.frame(Barcode = rownames(metadata1), CellType = metadata1$Annotated)  
colnames(cell_info1) <- c("Barcode_Celltype", "CellType")
head(cell_info1)

In [ ]:
cell_info1 <- cell_info1 %>% mutate(Barcode = sub(".*_", "", Barcode_Celltype))
cell_info1

In [ ]:
# Export to a CSV for use in the bash script
write.csv(cell_info1, 
          here(table_dir, paste0(params$sample_id, "_Diet_seurat_wnn-integrated_cellinfo.csv")), row.names = FALSE)